# Comparing Machine Learning Classification Methods

### Goal
Use **the same dataset and the same train/test split** to compare how different machine learning classifiers perform.

We will compare:

- Logistic Regression
- k-Nearest Neighbors (KNN)
- Support Vector Machine (SVM)
- Decision Tree
- Random Forest
- Gaussian Naive Bayes
- Multilayer Perceptron (MLP)

Dataset: **Breast Cancer Wisconsin** from `scikit-learn`.

The target is binary:

- `0` – malignant
- `1` – benign

> The important idea: **no single machine learning method is always the best**.  
> Performance depends on the dataset, preprocessing, hyperparameters, and evaluation metric.


In [ ]:
# If needed, uncomment this line:
# !pip install numpy pandas matplotlib scikit-learn

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

RANDOM_STATE = 42


## 1. Load the dataset

The dataset contains numerical measurements computed from digitized images of breast masses.

For this exercise, we treat it simply as a **binary classification dataset**.


In [ ]:
data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("X shape:", X.shape)
print("Classes:", dict(enumerate(data.target_names)))
print()
print("Class counts:")
print(y.value_counts().sort_index())

X.head()


## 2. Use exactly the same train/test split for every classifier

This is essential for a fair comparison.

`stratify=y` keeps approximately the same class proportions in the training and test sets.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))


## 3. Define the models

Some algorithms are sensitive to feature scale, so they are placed inside a pipeline with `StandardScaler`.

Tree-based methods do not require scaling.


In [ ]:
# Define all classifiers in one dictionary.
# Pipelines are used when a model benefits from feature standardization.
#
# StandardScaler():
#   transforms each feature approximately to mean = 0 and standard deviation = 1.
#   This is especially important for distance- or scale-sensitive methods
#   such as Logistic Regression, KNN, SVM, and MLP.

models = {

    # ------------------------------------------------------------
    # 1. LOGISTIC REGRESSION
    # ------------------------------------------------------------
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,              # Maximum number of optimization iterations.
                                        # A larger value gives the algorithm more time
                                        # to converge if the default is not sufficient.

            random_state=RANDOM_STATE   # Fixes the random seed for reproducibility.
                                        # With the same seed, stochastic parts of the
                                        # algorithm behave consistently between runs.
        ))
    ]),


    # ------------------------------------------------------------
    # 2. k-NEAREST NEIGHBORS (KNN)
    # ------------------------------------------------------------
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(
            n_neighbors=5               # Number of nearest training samples ("k")
                                        # used to classify a new observation.
                                        # Small k -> more flexible, more sensitive to noise.
                                        # Large k -> smoother decision boundary.
        ))
    ]),


    # ------------------------------------------------------------
    # 3. SUPPORT VECTOR MACHINE (SVM)
    # ------------------------------------------------------------
    "SVM (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",               # Kernel function.
                                        # "rbf" (Radial Basis Function) allows the SVM
                                        # to create nonlinear decision boundaries.

            probability=True,           # Enables predict_proba(), so class probabilities
                                        # can be estimated and ROC AUC can be calculated
                                        # using probability scores.

            random_state=RANDOM_STATE   # Fixes randomness used internally when
                                        # probability estimates are enabled.
        ))
    ]),


    # ------------------------------------------------------------
    # 4. DECISION TREE
    # ------------------------------------------------------------
    "Decision Tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE       # Makes random choices made while building
                                        # the tree reproducible.
    ),


    # ------------------------------------------------------------
    # 5. RANDOM FOREST
    # ------------------------------------------------------------
    "Random Forest": RandomForestClassifier(
        n_estimators=300,               # Number of Decision Trees in the forest.
                                        # More trees usually make predictions more stable,
                                        # but increase computation time.

        random_state=RANDOM_STATE       # Makes the random construction of the forest
                                        # reproducible between runs.
    ),


    # ------------------------------------------------------------
    # 6. GAUSSIAN NAIVE BAYES
    # ------------------------------------------------------------
    "Gaussian Naive Bayes": GaussianNB(),
    # No parameters are explicitly specified here.
    # Therefore, scikit-learn uses the default GaussianNB settings.
    # The method assumes that each feature follows a Gaussian (normal)
    # distribution within each class.


    # ------------------------------------------------------------
    # 7. MULTILAYER PERCEPTRON (MLP)
    # ------------------------------------------------------------
    "MLP Neural Network": Pipeline([
        ("scaler", StandardScaler()),
        ("model", MLPClassifier(
            hidden_layer_sizes=(50,),   # Architecture of the hidden layer(s).
                                        # (50,) means one hidden layer with 50 neurons.
                                        # For example, (50, 20) would mean two hidden
                                        # layers containing 50 and 20 neurons.

            max_iter=2000,              # Maximum number of training iterations (epochs
                                        # over the optimization procedure). A larger value
                                        # gives the optimizer more opportunities to converge.

            random_state=RANDOM_STATE   # Fixes the random initialization and other
                                        # stochastic operations for reproducibility.
        ))
    ])
}

print("Number of classifiers:", len(models))


## 4. Train and evaluate all classifiers

We calculate several metrics because **accuracy alone does not tell the whole story**.

- **Accuracy** – proportion of all correctly classified samples
- **Balanced accuracy** – average recall across classes
- **Precision** – how many predicted positives were correct
- **Recall** – how many actual positives were detected
- **F1-score** – balance between precision and recall
- **ROC AUC** – ranking/separation ability across classification thresholds


In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    start = time.perf_counter()

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    elapsed = time.perf_counter() - start

    # Probability / decision score for ROC AUC
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
    else:
        y_score = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_score),
        "Training + prediction time (s)": elapsed
    })

    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values(
    by="Balanced Accuracy",
    ascending=False
).reset_index(drop=True)

results_df.round(3)


## 5. Visual comparison

Notice that the ranking can change depending on which metric you use.


In [ ]:
plot_df = results_df.set_index("Model")[
    ["Accuracy", "Balanced Accuracy", "F1", "ROC AUC"]
]

ax = plot_df.plot(
    kind="bar",
    figsize=(12, 6),
    ylim=(0.70, 1.01)
)

ax.set_title("Performance of different classifiers on the same test set")
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.legend(loc="lower right")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 6. Confusion matrices

A confusion matrix shows **what kind of errors** each classifier makes.

This can be more informative than one summary number.


In [ ]:
for name, model in trained_models.items():
    ConfusionMatrixDisplay.from_estimator(
        model,
        X_test,
        y_test,
        display_labels=data.target_names,
        cmap="Blues"
    )
    plt.title(name)
    plt.tight_layout()
    plt.show()


## 7. ROC curves

ROC curves compare classifier performance over many possible decision thresholds.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name, model in trained_models.items():
    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        name=name,
        ax=ax
    )

ax.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
ax.set_title("ROC curves")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 8. Which classifier is the best?

Sort the table by different metrics.

You may find that a model that is best by one metric is not necessarily best by another.


In [ ]:
for metric in ["Accuracy", "Balanced Accuracy", "F1", "ROC AUC"]:
    best_row = results_df.loc[results_df[metric].idxmax()]
    print(f"{metric:18s}: {best_row['Model']} ({best_row[metric]:.3f})")


# Discussion questions for students

1. Why do different machine learning algorithms achieve different results on the same dataset?
2. Why do KNN, SVM, Logistic Regression, and MLP use `StandardScaler`, while Decision Tree and Random Forest do not?
3. Is the classifier with the highest **accuracy** automatically the best model?
4. Which error is more important in a medical classification problem: false positive or false negative?
5. Why can a Decision Tree overfit more easily than Logistic Regression?
6. Why can Random Forest perform better than a single Decision Tree?
7. What happens if you change `RANDOM_STATE`?
8. What happens if you change the size of the training set?
9. Would the same classifier still be the best on another dataset?


## 9. Student experiment

Try changing one parameter at a time:

```python
KNeighborsClassifier(n_neighbors=1)
KNeighborsClassifier(n_neighbors=15)

DecisionTreeClassifier(max_depth=2)
DecisionTreeClassifier(max_depth=None)

SVC(C=0.1)
SVC(C=100)

RandomForestClassifier(n_estimators=10)
RandomForestClassifier(n_estimators=500)
```

Then rerun the evaluation and observe how the results change.

### Main conclusion

> Machine learning performance is determined not only by the dataset, but also by the **choice of algorithm, preprocessing, hyperparameters, data split, and evaluation metric**.


# Part II — Comparing Regression Methods

In classification, the target is a **class label**.

In regression, the target is a **continuous numerical value**.

Here we will use the same idea as before:

- one dataset;
- one train/test split;
- several different regression algorithms;
- the same evaluation metrics for all algorithms.

We will compare:

- Linear Regression
- Ridge Regression
- k-Nearest Neighbors Regressor
- Support Vector Regression (SVR)
- Decision Tree Regressor
- Random Forest Regressor

Dataset: **Diabetes dataset** from `scikit-learn`.

The target is a continuous measure of disease progression one year after baseline.


In [ ]:
# Regression-specific imports

from sklearn.datasets import load_diabetes

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


## 1. Load the regression dataset

The input contains several numerical patient-related variables.

The target is **continuous**, so this is a regression problem.


In [ ]:
reg_data = load_diabetes()

X_reg = pd.DataFrame(reg_data.data, columns=reg_data.feature_names)
y_reg = pd.Series(reg_data.target, name="disease_progression")

print("X shape:", X_reg.shape)
print("Target range:", y_reg.min(), "to", y_reg.max())
print()
print(y_reg.describe())

X_reg.head()


## 2. Use the same train/test split for every regression model

Again, this is essential for a fair comparison.

All models will be trained on exactly the same training observations and tested on exactly the same test observations.


In [ ]:
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.30,              # 30% of observations are reserved for testing.
    random_state=RANDOM_STATE    # Makes the split reproducible.
)

print("Training samples:", len(X_reg_train))
print("Test samples:", len(X_reg_test))


## 3. Define the regression models

As in classification, some methods are sensitive to feature scale.

Therefore, `StandardScaler()` is used for:

- Ridge Regression
- KNN Regression
- Support Vector Regression

Tree-based methods generally do not require feature scaling.


In [ ]:
regression_models = {

    # ------------------------------------------------------------
    # 1. LINEAR REGRESSION
    # ------------------------------------------------------------
    "Linear Regression": LinearRegression(),
    # No parameters are explicitly specified here.
    # Linear Regression fits a straight-line / linear relationship
    # between the input features and the continuous target.


    # ------------------------------------------------------------
    # 2. RIDGE REGRESSION
    # ------------------------------------------------------------
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(
            alpha=1.0                   # Strength of L2 regularization.
                                        # Larger alpha -> stronger penalty on large
                                        # coefficients -> simpler model.
                                        # Smaller alpha -> closer to ordinary
                                        # Linear Regression.
        ))
    ]),


    # ------------------------------------------------------------
    # 3. k-NEAREST NEIGHBORS REGRESSOR
    # ------------------------------------------------------------
    "KNN Regressor": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(
            n_neighbors=5               # Number of nearest observations used
                                        # to predict the target value.
                                        # The prediction is usually based on the
                                        # average target value of these neighbors.
        ))
    ]),


    # ------------------------------------------------------------
    # 4. SUPPORT VECTOR REGRESSION (SVR)
    # ------------------------------------------------------------
    "SVR (RBF)": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(
            kernel="rbf",               # RBF kernel allows nonlinear relationships
                                        # between features and the target.

            C=1.0,                      # Controls the penalty for prediction errors.
                                        # Larger C -> tries harder to fit training data.
                                        # Smaller C -> stronger regularization.

            epsilon=0.1                 # Width of the epsilon-insensitive zone.
                                        # Errors smaller than epsilon are not penalized
                                        # in the SVR loss function.
        ))
    ]),


    # ------------------------------------------------------------
    # 5. DECISION TREE REGRESSOR
    # ------------------------------------------------------------
    "Decision Tree Regressor": DecisionTreeRegressor(
        max_depth=None,                 # Maximum depth of the tree.
                                        # None means the tree can grow until other
                                        # stopping rules are reached.
                                        # Very deep trees can overfit.

        random_state=RANDOM_STATE       # Makes tree construction reproducible.
    ),


    # ------------------------------------------------------------
    # 6. RANDOM FOREST REGRESSOR
    # ------------------------------------------------------------
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=300,               # Number of Decision Trees in the forest.
                                        # More trees usually improve stability,
                                        # but require more computation.

        random_state=RANDOM_STATE       # Makes the random forest reproducible.
    )
}

print("Number of regression models:", len(regression_models))


## 4. Train and evaluate all regression models

For regression we use different metrics than for classification:

- **MAE (Mean Absolute Error)**  
  Average absolute difference between predicted and true values.  
  **Lower is better.**

- **RMSE (Root Mean Squared Error)**  
  Similar to MAE, but large errors receive more weight.  
  **Lower is better.**

- **R² (Coefficient of Determination)**  
  Indicates how much of the variability in the target is explained by the model.  
  **Higher is better.**

Typical interpretation of R²:

- `R² = 1` → perfect predictions
- `R² = 0` → approximately no better than predicting the mean target
- `R² < 0` → worse than predicting the mean target


In [ ]:
reg_results = []
trained_regression_models = {}

for name, model in regression_models.items():
    start = time.perf_counter()

    model.fit(X_reg_train, y_reg_train)
    y_reg_pred = model.predict(X_reg_test)

    elapsed = time.perf_counter() - start

    mae = mean_absolute_error(y_reg_test, y_reg_pred)
    rmse = np.sqrt(mean_squared_error(y_reg_test, y_reg_pred))
    r2 = r2_score(y_reg_test, y_reg_pred)

    reg_results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2,
        "Training + prediction time (s)": elapsed
    })

    trained_regression_models[name] = model

reg_results_df = pd.DataFrame(reg_results).sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

reg_results_df.round(3)


## 5. Visual comparison of regression performance

Remember:

- lower **MAE** is better;
- lower **RMSE** is better;
- higher **R²** is better.


In [ ]:
ax = reg_results_df.set_index("Model")[["MAE", "RMSE"]].plot(
    kind="bar",
    figsize=(12, 6)
)

ax.set_title("Regression model errors on the same test set")
ax.set_ylabel("Error")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
ax = reg_results_df.set_index("Model")[["R²"]].plot(
    kind="bar",
    figsize=(10, 5),
    legend=False
)

ax.set_title("R² comparison")
ax.set_ylabel("R²")
ax.set_xlabel("")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## 6. Actual vs predicted values

A good regression model should produce points close to the diagonal line:

**predicted value ≈ actual value**


In [ ]:
for name, model in trained_regression_models.items():
    y_pred = model.predict(X_reg_test)

    plt.figure(figsize=(6, 5))
    plt.scatter(y_reg_test, y_pred, alpha=0.7)

    low = min(y_reg_test.min(), y_pred.min())
    high = max(y_reg_test.max(), y_pred.max())

    plt.plot([low, high], [low, high], linestyle="--")

    plt.xlabel("Actual target")
    plt.ylabel("Predicted target")
    plt.title(name)
    plt.tight_layout()
    plt.show()


## 7. Residual plots

A residual is:

**residual = actual value − predicted value**

Residuals close to zero indicate small prediction errors.

A useful residual plot can reveal whether the model systematically overpredicts or underpredicts certain observations.


In [ ]:
for name, model in trained_regression_models.items():
    y_pred = model.predict(X_reg_test)
    residuals = y_reg_test - y_pred

    plt.figure(figsize=(6, 5))
    plt.scatter(y_pred, residuals, alpha=0.7)
    plt.axhline(0, linestyle="--")

    plt.xlabel("Predicted target")
    plt.ylabel("Residual")
    plt.title(f"Residuals — {name}")
    plt.tight_layout()
    plt.show()


## 8. Which regression model is best?

Unlike classification, there is no Accuracy score here.

The ranking can also depend on the chosen regression metric.


In [ ]:
best_mae = reg_results_df.loc[reg_results_df["MAE"].idxmin()]
best_rmse = reg_results_df.loc[reg_results_df["RMSE"].idxmin()]
best_r2 = reg_results_df.loc[reg_results_df["R²"].idxmax()]

print(f"Best MAE : {best_mae['Model']} ({best_mae['MAE']:.3f})")
print(f"Best RMSE: {best_rmse['Model']} ({best_rmse['RMSE']:.3f})")
print(f"Best R²  : {best_r2['Model']} ({best_r2['R²']:.3f})")


# Classification vs Regression

| | Classification | Regression |
|---|---|---|
| **Target** | Class / category | Continuous number |
| **Example** | Disease / no disease | Blood pressure |
| **Typical methods** | Logistic Regression, KNN, SVM, Decision Tree, Random Forest, Naive Bayes | Linear Regression, Ridge, KNN Regressor, SVR, Decision Tree Regressor, Random Forest Regressor |
| **Typical metrics** | Accuracy, Precision, Recall, F1, ROC AUC | MAE, RMSE, R² |
| **Output** | Class or class probability | Numerical value |

The important point is that **many ML ideas appear in both classification and regression**, but the prediction target and evaluation metrics are different.


# Discussion questions

1. Why is Accuracy not appropriate for regression?
2. What is the difference between MAE and RMSE?
3. Why does RMSE penalize large errors more strongly than MAE?
4. What does an R² value close to 1 mean?
5. What does a negative R² value mean?
6. Why is scaling important for KNN and SVR?
7. Why can a Decision Tree Regressor overfit?
8. Why can Random Forest Regression be more stable than one Decision Tree?
9. Why might Linear Regression outperform a more complex nonlinear method on some datasets?
10. Would the best classifier necessarily also correspond to the best type of regression algorithm?


## Student experiment

Try changing one parameter at a time:

```python
KNeighborsRegressor(n_neighbors=1)
KNeighborsRegressor(n_neighbors=15)

DecisionTreeRegressor(max_depth=2)
DecisionTreeRegressor(max_depth=None)

SVR(C=0.1)
SVR(C=100)

RandomForestRegressor(n_estimators=10)
RandomForestRegressor(n_estimators=500)

Ridge(alpha=0.01)
Ridge(alpha=100)
```

Then rerun the evaluation and observe how **MAE, RMSE, and R²** change.

### Main conclusion

> Different regression algorithms can give substantially different predictions on the same dataset because they make different assumptions about the relationship between input features and the continuous target.
